# Zero-Shot Object Detection with Qwen3.5

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/QwenLM/Qwen3.5)

Open-vocabulary object detection using [Qwen3.5](https://github.com/QwenLM/Qwen3.5), Alibaba's natively multimodal model. Unlike traditional detectors (YOLO, RT-DETR) that need fixed class lists and training data, this approach uses natural language prompts to detect anything — named individuals, spatial relationships, actions, jersey numbers, colors, logos, and more.

**Examples covered in this notebook:**

- **Traffic scene** — detect cars, yellow taxis, multi-class vehicles
- **Solvay Conference photo** — detect named scientists (Einstein, Curie), spatial queries ("person between Einstein and Curie"), accessories (hats, canes)
- **Basketball game** — detect by jersey color, jersey number (#7), team name, action ("player about to shoot"), bench players, sponsor logos

## 1. Check GPU

In [9]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Sat Mar  7 16:17:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     On  |   00000000:01:00.0  On |                  N/A |
|  0%   39C    P8             25W /  300W |   14828MiB /  16303MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [2]:
!pip install -q 'git+https://github.com/huggingface/transformers.git@main'
!pip install -q supervision accelerate bitsandbytes
!pip install -q qwen-vl-utils torchvision pillow

## 3. Download Example Images

In [3]:
import urllib.request
import os

images = {
    "traffic_jam.jpg": "https://storage.googleapis.com/com-roboflow-marketing/notebooks/examples/traffic_jam.jpg",
    "solvay_conference.jpg": "https://storage.googleapis.com/com-roboflow-marketing/notebooks/examples/solvay_conference.jpg",
    "basketball_game.jpg": "https://storage.googleapis.com/com-roboflow-marketing/notebooks/examples/basketball_game.jpg",
}

for filename, url in images.items():
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded: {filename}")
    else:
        print(f"Already exists: {filename}")

Already exists: traffic_jam.jpg
Already exists: solvay_conference.jpg
Already exists: basketball_game.jpg


## 4. Load Qwen3.5 Model

In [ ]:
import os
os.environ["HF_TOKEN"] = ""

In [8]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch

MODEL_ID = "Qwen/Qwen3.5-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_ID} ...")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()

print(f"\n✅ Model loaded: {MODEL_ID}")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading Qwen/Qwen3.5-4B ...


Loading weights:  35%|███▌      | 256/723 [00:00<00:01, 310.02it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 46.00 MiB. GPU 0 has a total capacity of 15.48 GiB of which 58.19 MiB is free. Including non-PyTorch memory, this process has 15.04 GiB memory in use. Of the allocated memory 14.67 GiB is allocated by PyTorch, and 127.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. Helper Functions

In [ ]:
import re
import json
import torch
import numpy as np
from PIL import Image
import supervision as sv


def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks including partial/malformed ones."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)
    return text.strip()


def extract_json(text: str) -> list:
    """Parse Qwen3.5 output — handles both JSON array and numbered list formats."""
    text = strip_thinking(text)
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`").strip()

    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
            if isinstance(data, list) and data and isinstance(data[0], dict):
                return data
        except json.JSONDecodeError:
            pass

    pattern = re.compile(
        r'\d+\.\s*(?:[^:\n]+:\s*)?([^\[\n]+?)\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]'
    )

    results = []
    for m in pattern.finditer(text):
        label = m.group(1).strip().rstrip(":")
        x1, y1, x2, y2 = int(m.group(2)), int(m.group(3)), int(m.group(4)), int(m.group(5))
        results.append({"label": label, "bbox_2d": [x1, y1, x2, y2]})

    if results:
        print(f"  ℹ️  Parsed {len(results)} items from numbered list format")
        return results

    pattern2 = re.compile(r'([A-Za-z][^\[\n]{1,40}?)\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]')
    for m in pattern2.finditer(text):
        label = m.group(1).strip().strip(":.,-")
        x1, y1, x2, y2 = int(m.group(2)), int(m.group(3)), int(m.group(4)), int(m.group(5))
        results.append({"label": label, "bbox_2d": [x1, y1, x2, y2]})

    return results


def parse_detections(
    response: str,
    image_size: tuple,
    classes: list = None,
) -> sv.Detections:
    """Parse Qwen3.5 JSON output into sv.Detections."""
    W, H = image_size
    items = extract_json(response)

    if not items:
        print("  ⚠️  No JSON found in response")
        return sv.Detections.empty()

    xyxy_list, labels, class_ids = [], [], []

    for item in items:
        bbox = item.get("bbox_2d") or item.get("bbox") or item.get("box")
        label = item.get("label", "object")

        if bbox is None or len(bbox) != 4:
            continue

        x1, y1, x2, y2 = bbox

        if max(x1, y1, x2, y2) <= 1.0:
            x1, x2 = x1 * W, x2 * W
            y1, y2 = y1 * H, y2 * H
        else:
            x1, x2 = x1 / 1000 * W, x2 / 1000 * W
            y1, y2 = y1 / 1000 * H, y2 / 1000 * H

        x1 = max(0, min(x1, W))
        x2 = max(0, min(x2, W))
        y1 = max(0, min(y1, H))
        y2 = max(0, min(y2, H))

        if x2 <= x1 or y2 <= y1:
            continue

        xyxy_list.append([x1, y1, x2, y2])
        labels.append(label)

        if classes:
            label_lower = label.lower()
            cid = next(
                (i for i, c in enumerate(classes) if c.lower() in label_lower or label_lower in c.lower()),
                0
            )
            class_ids.append(cid)

    if not xyxy_list:
        print("  ⚠️  Parsed JSON but found no valid bounding boxes")
        return sv.Detections.empty()

    detections = sv.Detections(
        xyxy=np.array(xyxy_list, dtype=np.float32),
        class_id=np.array(class_ids) if class_ids else None,
        data={"class_name": np.array(labels)},
    )
    return detections


def qwen_detect(
    image: Image.Image,
    target: str,
    max_new_tokens: int = 4096,
    verbose: bool = False,
) -> str:
    prompt = (
        f"Detect all instances of: {target}. "
        f"Output ONLY a valid JSON array. "
        f"Each item must have exactly two fields: "
        f"'label' (string) and 'bbox_2d' ([x_min, y_min, x_max, y_max] in 0-1000 scale). "
        f"No explanation. No markdown. No extra text. Just the raw JSON array."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            top_k=20,
            do_sample=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    trimmed = [g[len(i):] for i, g in zip(inputs.input_ids, gen)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    if verbose:
        print("── Raw output ──────────────────────────────")
        print(raw[:500], "..." if len(raw) > 500 else "")
        print("────────────────────────────────────────────")

    return raw


COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff",
    "#ff66b2", "#ff8080", "#b266ff", "#9999ff",
    "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

def annotate_image(
    image: Image.Image,
    detections: sv.Detections,
    smart_position: bool = True,
) -> Image.Image:
    if len(detections) == 0:
        print("  ⚠️  No detections to annotate")
        return image

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
    color_by_class = detections.class_id is not None
    lookup = sv.ColorLookup.CLASS if color_by_class else sv.ColorLookup.INDEX

    box_annotator = sv.BoxAnnotator(color=COLOR, color_lookup=lookup, thickness=thickness)
    label_annotator = sv.LabelAnnotator(
        color=COLOR,
        color_lookup=lookup,
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        text_thickness=max(1, thickness - 1),
        smart_position=smart_position,
    )

    labels = list(detections.data.get("class_name", []))

    annotated = image.copy()
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)
    return annotated


def detect_and_show(
    image_path: str,
    target: str,
    classes: list = None,
    smart_position: bool = True,
    thumbnail_size: int = 900,
    verbose: bool = False,
) -> Image.Image:
    image = Image.open(image_path).convert("RGB")
    response = qwen_detect(image, target, verbose=verbose)

    print(f"Target     : '{target}'")

    detections = parse_detections(response, image.size, classes=classes)
    print(f"Detections : {len(detections)} found")

    if len(detections) > 0:
        is_multi_class = "," in target

        if not is_multi_class:
            detections.data["class_name"] = np.array([target] * len(detections))
        else:
            class_list = [c.strip().lower() for c in target.split(",")]
            clean_labels = []

            for raw_label in detections.data["class_name"]:
                raw_lower = raw_label.lower()
                matched = next(
                    (c for c in class_list if c in raw_lower or raw_lower in c),
                    raw_label
                )
                clean_labels.append(matched)

            detections.data["class_name"] = np.array(clean_labels)

    annotated = annotate_image(image, detections, smart_position=smart_position)
    annotated.thumbnail((thumbnail_size, thumbnail_size))
    return annotated


print("✅ Helper functions ready")

## 6. Detection Examples

### 6.1 Traffic Scene

In [ ]:
detect_and_show("traffic_jam.jpg", target="yellow taxi")

### 6.2 Solvay Conference — Named Person Detection & Spatial Reasoning

In [ ]:
detect_and_show(
    "solvay_conference.jpg",
    target="man, woman",
    classes=["man", "woman"],
    smart_position=False
)

In [ ]:
detect_and_show(
    "solvay_conference.jpg",
    target="albert einstein and marie curie",
    smart_position=False
)

In [ ]:
detect_and_show(
    "solvay_conference.jpg",
    target="person standing between albert einstein and marie curie",
    smart_position=False
)

In [ ]:
detect_and_show(
    "solvay_conference.jpg",
    target="people in front row",
    smart_position=False
)

In [ ]:
detect_and_show(
    "solvay_conference.jpg",
    target="hat, cane",
    classes=["hat", "cane"],
    smart_position=False
)

### 6.3 Basketball Game — Action, Jersey & Team Queries

In [ ]:
detect_and_show(
    "basketball_game.jpg",
    target="player in red, player in white",
    classes=["player in red", "player in white"],
    smart_position=False
)

In [ ]:
detect_and_show(
    "basketball_game.jpg",
    target="player #7",
    smart_position=False
)

In [ ]:
detect_and_show(
    "basketball_game.jpg",
    target="new york knicks player",
    smart_position=False
)

In [ ]:
detect_and_show(
    "basketball_game.jpg",
    target="player about to shoot",
    smart_position=False
)

In [ ]:
detect_and_show(
    "basketball_game.jpg",
    target="knicks player sitting on bench",
    smart_position=False
)

In [ ]:
detect_and_show(
    "basketball_game.jpg",
    target="sponsor logo",
    smart_position=False
)

## 7. Thinking Mode Comparison

Qwen3.5 supports a thinking mode where it reasons step-by-step before answering, which can improve accuracy on complex queries at the cost of speed.

In [ ]:
from PIL import Image
import time

image = Image.open("solvay_conference.jpg").convert("RGB")
target = "person standing between albert einstein and marie curie"

# Without thinking (fast)
print("=" * 50)
print("Mode: Non-thinking (fast)")
t0 = time.time()
result_fast = qwen_detect(image, target, thinking=False)
t1 = time.time()
print(f"Time: {t1-t0:.1f}s")
print(f"Output: {result_fast}\n")

# With thinking (slower but more precise)
print("=" * 50)
print("Mode: Thinking (accurate)")
t0 = time.time()
result_think = qwen_detect(image, target, thinking=True, verbose=True)
t1 = time.time()
print(f"Time: {t1-t0:.1f}s")
print(f"Output: {result_think}")

## 8. Try Your Own Image

In [ ]:
YOUR_IMAGE = "your_image.jpg"
YOUR_TARGET = "person wearing helmet"

if os.path.exists(YOUR_IMAGE):
    result = detect_and_show(YOUR_IMAGE, target=YOUR_TARGET)
    display(result)
else:
    print(f"Image not found: {YOUR_IMAGE}")
    print("Set YOUR_IMAGE to a valid path and re-run.")

## 9. Batch Processing

In [ ]:
import torch

def run_batch(queries: list[tuple[str, str]], thinking: bool = False) -> dict:
    """Run multiple image+target pairs and return results dict."""
    results = {}

    for i, (img_path, target) in enumerate(queries):
        image = Image.open(img_path).convert("RGB")
        vram_before = torch.cuda.memory_allocated() / 1e9

        response = qwen_detect(image, target, thinking=thinking)

        vram_after = torch.cuda.memory_allocated() / 1e9
        print(f"[{i+1}/{len(queries)}] '{target}' on {img_path}")
        print(f"  VRAM: {vram_before:.2f} → {vram_after:.2f} GB")
        print(f"  Response: {response[:120]}...\n")

        results[(img_path, target)] = response
        torch.cuda.empty_cache()

    return results


queries = [
    ("traffic_jam.jpg",       "yellow taxi"),
    ("solvay_conference.jpg", "man, woman"),
    ("basketball_game.jpg",   "player about to shoot"),
]

batch_results = run_batch(queries)

## 10. Cleanup

In [ ]:
import gc

def cleanup():
    global model, processor
    del model
    del processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM freed. Remaining: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# cleanup()

---

## Summary

| Capability | Traditional Detectors | Qwen3.5 |
|---|---|---|
| Fixed class detection | Fast (30-200 FPS) | Slower (0.5-3 FPS) |
| Open vocabulary queries | No | Yes |
| Named person detection | No | Yes |
| Spatial relationship queries | No | Yes |
| Action/pose queries | No | Yes |
| Jersey number reading | No | Yes |
| No training required | Yes | Yes |

Qwen3.5 is best suited for zero-shot detection on new domains, dataset annotation, and tasks where the class vocabulary isn't known in advance.